# Page 5 — "How to Reason": the prompt ladder

Five prompt formulations on one model, measuring what each buys — from a forced blurt to
few-shot chain-of-thought. Everything page 5 renders is produced here; the page replays
committed logs and runs no model.

**The one constraint: thinking is OFF for every generation.** The prompt is the only variable —
the *Proof that thinking is off* cell shows the rendered template, and the integrity cell
asserts no trace ever contained thinking content.

Writes to `out/page_05/`: `traces.jsonl` (every generation, published as a download),
`derived.json` (what the page plots, bundled), `manifest.json` (provenance).

All rationale — the rung design, the item set, what the page may and may not claim — lives in
[`docs/page_5_refactor.md`](../docs/page_5_refactor.md).

## How to run this

Two modes, set by one variable in the next cell.

| Mode | Model | Items | Cap | Where it finishes |
|---|---|---|---|---|
| `QUICK` | Qwen3-0.6B | 12 eval + 8 held-out | 512 tokens | a laptop CPU, ~30 min |
| `FULL` | Qwen3-4B | 160 eval + 24 held-out | 1024 tokens | a 16 GB GPU, ~2 hours |

Runtimes are estimates until the approval run measures them. `QUICK` is the same pipeline with
noisier bars; the page ships `FULL`. The run is resumable: `traces.jsonl` is appended as it
goes, and re-running skips work already recorded. Run from this notebook's directory (Jupyter's
default) — outputs land in `out/page_05/` relative to the working directory.

```bash
pip install "torch>=2.7" "transformers>=4.56" accelerate sympy matplotlib requests
pip install reasoning-from-scratch   # or: pip install git+https://github.com/rasbt/reasoning-from-scratch.git
```

In [ ]:
# ---------------------------------------------------------------------------
# The two knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "QUICK"        # "QUICK" (laptop CPU, ~30 min) or "FULL" (GPU, ~2 h)
SEED = 123                # only selects items; decoding is greedy and unseeded

PRESETS = {
    "QUICK": dict(
        model_id="Qwen/Qwen3-0.6B",
        n_eval=12,                     # evaluated items
        n_pool=8,                      # held-out pool the exemplars are harvested from
        few_shot_k=2,
        max_new_tokens=512,
    ),
    "FULL": dict(
        model_id="Qwen/Qwen3-4B",
        n_eval=160,
        n_pool=24,
        few_shot_k=3,
        max_new_tokens=1024,
    ),
}
CFG = PRESETS[RUN_MODE]

# One generation cap for every rung. This page's variable is the PROMPT, so the
# length budget must not also vary -- that is page 6's experiment. The cap is set
# generously so it should rarely bind; the sweep reports the truncation rate, and
# if it is above a few percent the cap is too low and should be raised.
MAX_NEW_TOKENS = CFG["max_new_tokens"]

# Fallback ladder if the model does not fit: 4B -> 1.7B -> 0.6B. All three are
# hybrid Qwen3 checkpoints with the same tokenizer, so token counts stay
# comparable across them.
#   Qwen3-4B    ~8.0 GB bf16
#   Qwen3-1.7B  ~3.4 GB bf16
#   Qwen3-0.6B  ~1.2 GB bf16

print(f"mode={RUN_MODE}  model={CFG['model_id']}  "
      f"items={CFG['n_eval']} (+{CFG['n_pool']} held out)  cap={MAX_NEW_TOKENS}")

In [ ]:
import hashlib
import json
import random
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# The inherited harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_05")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = lambda xs: sum(xs) / max(len(xs), 1)

print("torch", torch.__version__)

## The grading harness

Inherited unchanged from ch03: `extract_final_candidate` pulls the last `\boxed{...}` out of a
completion (falling back to the last number), and `grade_answer` checks equivalence with light
LaTeX canonicalisation and SymPy. The next cell asserts it behaves as the sweep assumes,
including the "wrote a chain but boxed the answer" case the CoT rungs depend on.

In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                          "9",              True),
    (r"Step 1: 23-20=3. Step 2: 3+6=9. So \boxed{9}.",     "9",              True),   # a chain
    (r"27",                                                 "9",              False),  # a wrong blurt
    (r"\boxed{\frac{1}{2}}",                               "0.5",            True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                     "1024",           True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",           r"\left( 3, \frac{\pi}{2} \right)", True),
    (r"I am not sure.",                                     "9",              False),  # no answer at all
]

for text, truth, expected in CHECKS:
    got = bool(grade_answer(extract_final_candidate(text), truth))
    status = "ok " if got == expected else "FAIL"
    print(f"{status} extract={extract_final_candidate(text)!r:<40} truth={truth!r:<12} -> {got}")
    assert got == expected, f"grader behaved unexpectedly on {text!r}"

print("\nharness behaves as the sweep assumes.")

## The item set

MATH-500 filtered to levels 4 and 5: the full set sits near a maths-tuned model's ceiling, and
a ladder whose every rung scores 90% measures nothing.

**The eval set and the held-out exemplar pool are disjoint by construction** — otherwise the
few-shot rungs would be scored on problems whose worked answers appear in their own prompt.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

hard = [r for r in math500 if r["level"] in (4, 5)]
hard.sort(key=lambda r: r["unique_id"])          # deterministic before shuffling

rng = random.Random(SEED)
order = list(range(len(hard)))
rng.shuffle(order)

pool_items = [hard[i] for i in order[:CFG["n_pool"]]]
eval_items = [hard[i] for i in order[CFG["n_pool"]:CFG["n_pool"] + CFG["n_eval"]]]

assert not ({r["unique_id"] for r in pool_items} & {r["unique_id"] for r in eval_items}), \
    "exemplar pool leaked into the eval set"


def sha256_of(strings):
    h = hashlib.sha256()
    for s in strings:
        h.update(s.encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


ITEM_SET_HASH = sha256_of(r["unique_id"] for r in eval_items)

lv = lambda items, n: sum(r["level"] == n for r in items)
print(f"MATH-500 levels 4+5: {len(hard)} available")
print(f"  eval set    {len(eval_items):>4}  (L4={lv(eval_items,4)}, L5={lv(eval_items,5)})  sha256 {ITEM_SET_HASH[:16]}...")
print(f"  held-out    {len(pool_items):>4}  (L4={lv(pool_items,4)}, L5={lv(pool_items,5)})")
print(f"\nexample item:\n  {eval_items[0]['problem'][:200]}")
print(f"  answer: {eval_items[0]['answer']}")

In [ ]:
device = get_device()

# bf16 on GPU keeps quantisation out of our numbers (on Apple MPS it needs the
# torch >= 2.7 the install line pins); fp32 on CPU, where bf16 is slow and
# patchily supported. The dtype actually used is recorded in the manifest.
DTYPE = torch.float32 if device.type == "cpu" else torch.bfloat16

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], dtype=DTYPE, device_map=None,
).to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {CFG['model_id']}  {n_params/1e9:.2f}B params  {DTYPE}  on {device}  "
      f"({time.time()-t0:.1f}s)")

## Proof that thinking is off

Qwen3's chat template takes an `enable_thinking` flag; **off**, it pre-closes the think block
(`<think>\n\n</think>`), so the model has nowhere to put a chain. Every generation here uses
`enable_thinking=False`. The next cell prints both renderings so the difference is visible,
and the integrity cell asserts that no trace ever contained thinking content.

In [ ]:
demo_msgs = [{"role": "user", "content": "What is 2+2?"}]

off = tokenizer.apply_chat_template(demo_msgs, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
on = tokenizer.apply_chat_template(demo_msgs, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=True)

print("=== enable_thinking=False  <- every generation in this notebook ===")
print(repr(off))
print("\n=== enable_thinking=True   <- never used here, shown for contrast ===")
print(repr(on))
print("\ndifference:", "the OFF template pre-closes the think block" if off != on
      else "NONE -- this tokenizer has no thinking toggle, see note below")

# If a checkpoint has no toggle at all (a non-hybrid model), the page's constraint
# still holds -- thinking is off because the model has no thinking mode -- but say
# so in the manifest rather than implying we disabled something.
HAS_THINKING_TOGGLE = off != on

## The generation loop

The only place a model is called. Two recorded details a re-deriver should not drop:
**token strings**, decoded incrementally so `"".join(tokens) == text` exactly (asserted),
because the page's reveal steps through real token boundaries; and **`truncated`**, because on
this page the cap binding is a defect rather than a variable, and the sweep reports the rate.

In [ ]:
def incremental_token_strings(ids):
    '''Decode ids one at a time, but incrementally, so multi-byte characters
    split across tokens are not mangled. Concatenates back to the full text.'''
    pieces, prev = [], ""
    for k in range(1, len(ids) + 1):
        cur = tokenizer.decode(ids[:k], skip_special_tokens=True)
        pieces.append(cur[len(prev):])
        prev = cur
    return pieces


@torch.inference_mode()
def generate(prompt_text, mdl=None, tok=None, max_new_tokens=None):
    mdl = mdl or model
    tok = tok or tokenizer
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS

    chat = tok.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,          # <- load-bearing; see "Proof that thinking is off"
    )
    enc = tok(chat, return_tensors="pt").to(mdl.device)

    out = mdl.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,                            # greedy: no temperature, no top-p
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    new_ids = out[0][enc["input_ids"].shape[1]:].tolist()

    text = tok.decode(new_ids, skip_special_tokens=True)
    tokens = incremental_token_strings(new_ids)
    assert "".join(tokens) == text, "token strings do not reconstruct the text"

    return {
        "text": text,
        "tokens": tokens,
        "tokens_generated": len(new_ids),
        "truncated": len(new_ids) >= max_new_tokens,
        "thinking_leaked": "<think>" in text,
    }


smoke = generate("What is 2+2? Reply with just the number.", max_new_tokens=32)
print(f"{smoke['tokens_generated']} tokens, truncated={smoke['truncated']}, "
      f"thinking_leaked={smoke['thinking_leaked']}")
print("tokens:", smoke["tokens"][:12])
print("text:", smoke["text"][:200])

## The ladder

Five formulations, from least to most instruction. Rung design rationale is in the spec, §§2–3.

| Rung | What is added | Whose mechanism |
|---|---|---|
| `bare` | nothing — the harness prompt alone | the true default (ch03's baseline) |
| `forced_blurt` | "Give only the final answer. Do not explain." | the floor |
| `zero_shot_cot` | `"\n\nExplain step by step."` | Kojima et al. 2022 · ch04 §4.3 |
| `one_shot` | one worked example prepended | Wei et al. 2022 |
| `few_shot` | k worked examples prepended | Wei et al. 2022 |

**Every rung, including `bare`, carries ch03's `render_prompt`, which fixes the `\boxed{}`
output format.** Vary that too and the ladder is confounded — a rung could score lower only
because the grader cannot find its answer. The exemplar rungs get no "explain step by step":
combining Wei's and Kojima's mechanisms would measure a third thing.

In [ ]:
RUNGS = [
    ("bare",          "bare question"),
    ("forced_blurt",  "\"answer with the number\""),
    ("zero_shot_cot", "\"explain step by step\""),
    ("one_shot",      "one worked example"),
    ("few_shot",      f"{CFG['few_shot_k']} worked examples"),
]
RUNG_KEYS = [k for k, _ in RUNGS]
RUNG_LABEL = dict(RUNGS)

FORCED_BLURT_SUFFIX = "\n\nGive only the final answer. Do not explain."
COT_SUFFIX = "\n\nExplain step by step."      # ch04 section 4.3, verbatim mechanism


def build_prompt(rung, question, exemplars=None):
    '''The whole ladder, in one readable function. Nothing else varies.'''
    base = render_prompt(question)             # ch03's template -- CONSTANT across rungs

    if rung == "bare":
        return base
    if rung == "forced_blurt":
        return base + FORCED_BLURT_SUFFIX
    if rung == "zero_shot_cot":
        return base + COT_SUFFIX

    k = 1 if rung == "one_shot" else CFG["few_shot_k"]
    shots = "".join(
        render_prompt(e["question"]) + "\n" + e["trace"].strip() + "\n\n"
        for e in (exemplars or [])[:k]
    )
    return shots + base


# The diff that figure 1 highlights: bare -> zero-shot CoT is exactly these words.
print(f"the {len(COT_SUFFIX.split())} words the page highlights:", repr(COT_SUFFIX.strip()))
print()
print("--- bare " + "-" * 60)
print(build_prompt("bare", "What is 23 - 20 + 6?"))
print("\n--- zero_shot_cot " + "-" * 52)
print(build_prompt("zero_shot_cot", "What is 23 - 20 + 6?"))

## The sweep

One pass per rung over the eval set; the per-item record is the `rec` dict below — the
trace-bank schema shared with pages 6 through 9, with `rung` in place of page 6's `budget`.
Provenance lives once in `manifest.json`. The sweep appends to `traces.jsonl` and skips
`(rung, problem_id)` pairs already present, so an interrupted `FULL` run resumes.

In [ ]:
RUN_ID = f"{RUN_MODE.lower()}-{CFG['model_id'].split('/')[-1]}-{ITEM_SET_HASH[:8]}"


def load_done():
    '''(rung, problem_id) -> record, for resume.'''
    done = {}
    if TRACES_PATH.exists():
        with TRACES_PATH.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r.get("run_id") == RUN_ID:
                    done[(r["rung"], r["problem_id"])] = r
    return done


DONE = load_done()
print(f"run_id={RUN_ID}   {len(DONE)} generations already on disk")


def append_trace(rec):
    with TRACES_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def run_rung(rung, items, exemplars=None, mdl=None, tok=None, label=None, split="eval"):
    records, t_start = [], time.time()
    exemplar_ids = [e["problem_id"] for e in (exemplars or [])][
        : (1 if rung == "one_shot" else CFG["few_shot_k"])
    ] if rung in ("one_shot", "few_shot") else []

    for i, row in enumerate(items, start=1):
        key = (rung, row["unique_id"])
        if key in DONE:
            records.append(DONE[key])
            continue

        prompt = build_prompt(rung, row["problem"], exemplars)
        g = generate(prompt, mdl=mdl, tok=tok)
        extracted = extract_final_candidate(g["text"])

        rec = {
            "run_id": RUN_ID,
            "problem_id": row["unique_id"],
            "split": split,                    # "eval" is scored; "held_out" only feeds exemplars
            "level": row["level"],
            "subject": row["subject"],
            "question": row["problem"],
            "ground_truth": row["answer"],
            "rung": rung,
            "prompt": prompt,
            "trace": g["text"],
            "tokens": g["tokens"],
            "truncated": g["truncated"],
            "extracted_answer": extracted,
            "correct": bool(grade_answer(extracted, row["answer"])),
            "tokens_generated": g["tokens_generated"],
            "exemplar_ids": exemplar_ids,
            "thinking_leaked": g["thinking_leaked"],
        }
        append_trace(rec)
        DONE[key] = rec
        records.append(rec)

        elapsed = time.time() - t_start
        eta = elapsed / i * (len(items) - i)
        print(f"\r{label or rung}: {i}/{len(items)}  "
              f"acc={sum(r['correct'] for r in records)/len(records):.2f}  "
              f"eta {eta/60:.1f}m   ", end="")

    print()
    return records

In [ ]:
# Phase A: the three rungs that need no exemplars. zero_shot_cot also runs on the
# held-out pool -- that pass is what the worked examples are harvested from.
results = {}
for rung in ("bare", "forced_blurt", "zero_shot_cot"):
    results[rung] = run_rung(rung, eval_items, label=rung)

pool_cot = run_rung("zero_shot_cot", pool_items, label="pool/zero_shot_cot", split="held_out")


# The framing verdict (spec §3): does the bare rung already show working? Decides
# figure 1's left-hand rung and the page's verb ("make it reason" vs "let it
# reason"); recorded in derived.json so it cannot later be reversed in prose.
def shows_working(rec):
    lines = [ln for ln in rec["trace"].strip().splitlines() if ln.strip()]
    return len(lines) > 2 or rec["tokens_generated"] > 60


bare_working_share = mean([shows_working(r) for r in results["bare"]])
BARE_ALREADY_REASONS = bare_working_share >= 0.5
FRAMING_VERB = "let it reason" if BARE_ALREADY_REASONS else "make it reason"
FIGURE1_LEFT = "forced_blurt" if BARE_ALREADY_REASONS else "bare"

print(f"\nbare rung shows working on {bare_working_share:.0%} of items")
print(f'-> page verb: "{FRAMING_VERB}"   figure 1 left: {FIGURE1_LEFT}')

## Harvesting the worked examples

**The exemplars are the model's own verified-correct solutions to held-out problems** —
authored exemplars would be the one thing on the page a reader could not re-derive (spec §3).
Selection is a deterministic filter: correct under the grader, untruncated, short enough to
fit, shortest first with ties broken by `problem_id`.

In [ ]:
MAX_EXEMPLAR_TOKENS = 400

candidates = [
    r for r in pool_cot
    if r["correct"] and not r["truncated"] and r["tokens_generated"] <= MAX_EXEMPLAR_TOKENS
]
candidates.sort(key=lambda r: (r["tokens_generated"], r["problem_id"]))

if not candidates:
    raise RuntimeError(
        "No usable exemplars: the model solved nothing in the held-out pool (correct, "
        "untruncated, <= MAX_EXEMPLAR_TOKENS). Raise n_pool or MAX_EXEMPLAR_TOKENS and "
        "re-run, or run FULL with a larger model. Do not author exemplars by hand -- the "
        "page's claim is that they are the model's own verified work."
    )

exemplars = [
    {
        "problem_id": r["problem_id"],
        "level": r["level"],
        "question": r["question"],
        "trace": r["trace"],
        "ground_truth": r["ground_truth"],
        "extracted_answer": r["extracted_answer"],
        "tokens_generated": r["tokens_generated"],
    }
    for r in candidates[: CFG["few_shot_k"]]
]

if len(exemplars) < CFG["few_shot_k"]:
    print(f"WARNING: only {len(exemplars)} usable exemplars from a pool of {len(pool_items)}.\n"
          f"         Raise n_pool, or MAX_EXEMPLAR_TOKENS, and re-run. The few-shot rung\n"
          f"         below will use however many were found, which weakens the comparison.")

print(f"{len(candidates)} of {len(pool_items)} held-out problems were solved correctly, "
      f"untruncated, and short enough.\n")
for e in exemplars:
    print(f"  {e['problem_id']:<34} L{e['level']}  {e['tokens_generated']:>4} tokens  "
          f"-> {e['extracted_answer']}")

print("\n--- the first exemplar, exactly as it will be prepended " + "-" * 20)
print(render_prompt(exemplars[0]["question"]))
print(exemplars[0]["trace"].strip()[:800])

In [ ]:
# Phase B: the exemplar rungs. Their prompts contain no instruction to reason --
# only examples of reasoning.
for rung in ("one_shot", "few_shot"):
    results[rung] = run_rung(rung, eval_items, exemplars=exemplars, label=rung)

## Sweep integrity

Three assertions the page's honesty rests on. If any of these fails, the numbers below do not
mean what the page says they mean.

In [ ]:
all_records = [r for rung in RUNG_KEYS for r in results[rung]]

leaked = [r for r in all_records if r["thinking_leaked"]]
print(f"1. thinking never leaked into a trace ......... {len(leaked)} violations")
assert not leaked, "a trace contained a <think> block; the page's core constraint is broken"

trunc_rate = sum(r["truncated"] for r in all_records) / len(all_records)
print(f"2. the generation cap rarely bound ............ {trunc_rate:.1%} truncated")
if trunc_rate > 0.05:
    print("   ^ above 5%: raise MAX_NEW_TOKENS and re-run. Page 5 lets every prompt run to")
    print("     natural length; capping it is page 6's experiment, not this one.")

ex_ids = {e["problem_id"] for e in exemplars}
overlap = ex_ids & {r["problem_id"] for r in results["few_shot"]}
print(f"3. exemplars stayed out of the eval set ....... {len(overlap)} overlaps")
assert not overlap, "an exemplar problem was also scored"

print(f"\n{len(all_records)} generations across {len(RUNG_KEYS)} rungs and "
      f"{len(eval_items)} items.")

## Figure 1 — the specimen pair

**Curated from the sweep, not generated for the figure:** search the run for a problem the
left-hand rung (chosen by the framing verdict above) misses and `zero_shot_cot` lands, sorted
by chain length — legibility, not existence, is the binding constraint. The prompt diff at the
end is the highlight the page draws; the whole argument is in it.

In [ ]:
MAX_SPECIMEN_TOKENS = 260      # a chain the reader can walk on screen

by_id = {r["problem_id"]: r for r in results[FIGURE1_LEFT]}
pairs = []
for right in results["zero_shot_cot"]:
    left = by_id.get(right["problem_id"])
    if left and not left["correct"] and right["correct"] and not right["truncated"]:
        pairs.append((left, right))
pairs.sort(key=lambda p: (p[1]["tokens_generated"], p[1]["problem_id"]))

if not pairs:
    raise RuntimeError(
        "No specimen pair found. In QUICK mode with 12 items this can happen by chance -- "
        "raise n_eval, or run FULL. Do not hand-pick or author a pair."
    )
left_spec, right_spec = next(
    (p for p in pairs if p[1]["tokens_generated"] <= MAX_SPECIMEN_TOKENS), pairs[0]
)

# The prompt diff the page highlights.
shared_stem = left_spec["prompt"].replace(FORCED_BLURT_SUFFIX, "")
assert right_spec["prompt"].startswith(shared_stem), \
    "the two prompts do not share a common stem; the figure's premise fails"
PROMPT_DIFF = {
    "shared": shared_stem,
    "left_added": left_spec["prompt"][len(shared_stem):],
    "right_added": right_spec["prompt"][len(shared_stem):],
}

print(f"{len(pairs)} candidate pairs; chosen {right_spec['problem_id']} "
      f"(L{right_spec['level']}, chain = {right_spec['tokens_generated']} tokens)\n")
print("QUESTION:", right_spec["question"][:300])
print("TRUTH   :", right_spec["ground_truth"])
print(f"\n--- {FIGURE1_LEFT}: answered {left_spec['extracted_answer']!r} [WRONG], "
      f"{left_spec['tokens_generated']} tokens ---")
print(left_spec["trace"].strip()[:600])
print(f"\n--- zero_shot_cot: answered {right_spec['extracted_answer']!r} [RIGHT], "
      f"{right_spec['tokens_generated']} tokens ---")
print(right_spec["trace"].strip()[:1200])
print("\nleft adds :", repr(PROMPT_DIFF["left_added"]))
print("right adds:", repr(PROMPT_DIFF["right_added"]))
print(f"-> {len(PROMPT_DIFF['right_added'].split())} words separate a "
      f"{left_spec['tokens_generated']}-token miss from a "
      f"{right_spec['tokens_generated']}-token hit.")

## Figure 2 — the ladder, measured

Horizontal bars, one per rung; the cost row underneath is the handoff to page 6. **Model size
appears on no axis** (spec §1).

In [ ]:
def summarise(rung):
    rs = results[rung]
    toks = sorted(r["tokens_generated"] for r in rs)
    return {
        "rung": rung,
        "label": RUNG_LABEL[rung],
        "n": len(rs),
        "n_correct": sum(r["correct"] for r in rs),
        "accuracy": mean([r["correct"] for r in rs]),
        "mean_tokens": mean(toks),
        "median_tokens": toks[len(toks) // 2],
        "total_tokens": sum(toks),
        "truncated": sum(r["truncated"] for r in rs),
    }


summary = [summarise(k) for k in RUNG_KEYS]
acc = {s["rung"]: s["accuracy"] for s in summary}

print(f"{'rung':<16}{'prompt':<26}{'solved':>10}{'accuracy':>10}{'mean tok':>10}")
print("-" * 72)
for s in summary:
    print(f"{s['rung']:<16}{s['label']:<26}{s['n_correct']:>4}/{s['n']:<5}"
          f"{s['accuracy']:>9.1%}{s['mean_tokens']:>10.0f}")

best = max(summary, key=lambda s: s["accuracy"])
floor = min(summary, key=lambda s: s["accuracy"])

# The page's caption is drafted against these, not against expectations.
ordered = [s["rung"] for s in sorted(summary, key=lambda s: s["accuracy"])]
expected = ["forced_blurt", "bare", "zero_shot_cot", "one_shot", "few_shot"]
collapse = abs(acc["few_shot"] - acc["one_shot"]) * 100 < 2.0

print(f"\nranking, worst to best: {' < '.join(ordered)}"
      + ("" if ordered == expected
         else "  (differs from the naive expectation -- the caption should say so)"))
if collapse:
    print("one_shot and few_shot within 2 points -> consider collapsing them into one rung")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.6))
ys = range(len(summary))[::-1]
ax.barh(list(ys), [s["accuracy"] * 100 for s in summary], height=0.55, color="#3b4a6b")
ax.set_yticks(list(ys))
ax.set_yticklabels([s["label"] for s in summary])
ax.set_xlabel("share of problems solved (%)")
ax.set_xlim(0, 100)
ax.spines[["top", "right"]].set_visible(False)

for y, s in zip(ys, summary):
    ax.text(s["accuracy"] * 100 + 1.2, y, f"{s['accuracy']*100:.0f}%",
            va="center", fontsize=9)

cost = "  ·  ".join(f"{s['mean_tokens']:.0f}" for s in summary)
ax.set_title(f"{CFG['model_id']} · thinking off · greedy · MATH-500 L4-5, n={len(eval_items)}",
             fontsize=10, loc="left")
fig.text(0.01, -0.04, f"tokens written, same order:  {cost}", fontsize=9, color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2.png", dpi=160, bbox_inches="tight")
plt.show()

## The close

One paired readout, not a third chart: the best rung's accuracy beside what it cost. This is
the number page 6 opens on.

In [ ]:
close = {
    "best_rung": best["rung"],
    "best_label": best["label"],
    "best_accuracy": best["accuracy"],
    "best_mean_tokens": best["mean_tokens"],
    "floor_rung": floor["rung"],
    "floor_label": floor["label"],
    "floor_accuracy": floor["accuracy"],
    "floor_mean_tokens": floor["mean_tokens"],
    "accuracy_gain_pts": (best["accuracy"] - floor["accuracy"]) * 100,
    "token_multiple": best["mean_tokens"] / max(floor["mean_tokens"], 1e-9),
}
print(f"{close['best_label']:<26} {close['best_accuracy']:>6.1%}   "
      f"{close['best_mean_tokens']:>6.0f} tokens/problem")
print(f"{close['floor_label']:<26} {close['floor_accuracy']:>6.1%}   "
      f"{close['floor_mean_tokens']:>6.0f} tokens/problem")
print(f"\n{close['accuracy_gain_pts']:+.1f} points of accuracy, "
      f"for {close['token_multiple']:.0f}x the tokens.")

## Artifacts

`derived.json` is the contract between this notebook and the page: the page reads it and
nothing else, and **no inference runs at demonstration time.** It carries the aggregates, the
figure 1 specimens with their token strings, the harvested exemplars plus the exact few-shot
prompt for the specimen problem (the "see full prompt" disclosure renders it verbatim), and
the framing verdict. `traces.jsonl` stays out of the web bundle and is published as a download
beside the notebook link.

In [ ]:
def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None


def repo_revision(model_id):
    try:
        from huggingface_hub import model_info
        return model_info(model_id).sha
    except Exception:
        return None


def tokenizer_fingerprint(tok):
    try:
        return hashlib.sha256(tok.backend_tokenizer.to_str().encode()).hexdigest()
    except Exception:
        return None


manifest = {
    "run_id": RUN_ID,
    "notebook": "page_05_how_to_reason.ipynb",
    "page": 5,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": git_sha(),
    "model": {
        "id": CFG["model_id"],
        "revision": repo_revision(CFG["model_id"]),
        "tokenizer_sha256": tokenizer_fingerprint(tokenizer),
        "params": n_params,
        "dtype": str(DTYPE),
        "thinking": "off",
        "has_thinking_toggle": HAS_THINKING_TOGGLE,
    },
    "decoding": {"strategy": "greedy", "do_sample": False,
                 "max_new_tokens": MAX_NEW_TOKENS},
    "items": {
        "source": "MATH-500 (ch03 copy)",
        "filter": "level in {4, 5}",
        "n_eval": len(eval_items),
        "n_held_out": len(pool_items),
        "eval_set_sha256": ITEM_SET_HASH,
        "seed": SEED,
    },
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "chat_template_thinking_off": off,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device)},
    "integrity": {"thinking_leaks": len(leaked), "truncation_rate": trunc_rate},
}

# The exact few-shot prompt sent for the specimen problem, straight from the sweep
# records -- the page's "see full prompt" disclosure renders this verbatim instead
# of re-assembling it in TypeScript.
few_shot_prompt = next(
    r["prompt"] for r in results["few_shot"] if r["problem_id"] == right_spec["problem_id"]
)

derived = {
    "manifest": manifest,
    "framing": {
        "bare_rung_already_reasons": BARE_ALREADY_REASONS,
        "bare_working_share": bare_working_share,
        "verb": FRAMING_VERB,
        "figure1_left_rung": FIGURE1_LEFT,
    },
    "figure_1": {
        "left": {k: left_spec[k] for k in
                 ("rung", "trace", "tokens", "tokens_generated", "extracted_answer", "correct")},
        "right": {k: right_spec[k] for k in
                  ("rung", "trace", "tokens", "tokens_generated", "extracted_answer", "correct")},
        "problem_id": right_spec["problem_id"],
        "level": right_spec["level"],
        "question": right_spec["question"],
        "ground_truth": right_spec["ground_truth"],
        "prompt_diff": PROMPT_DIFF,
        "n_candidate_pairs": len(pairs),
    },
    "figure_2": {
        "rungs": summary,
        "ranking_worst_to_best": ordered,
        "matches_naive_expectation": ordered == expected,
        "one_shot_few_shot_within_2pts": collapse,
    },
    "close": close,
    "exemplars": exemplars,
    "few_shot_prompt_example": {
        "problem_id": right_spec["problem_id"],
        "prompt": few_shot_prompt,
    },
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

kb = lambda p: p.stat().st_size / 1024
print(f"derived.json   {kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
print(f"manifest.json  {kb(MANIFEST_PATH):>9.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {kb(TRACES_PATH):>9.1f} KB   <- published as a download")

## Notes for re-derivation

- Greedy decoding can diverge on a handful of items across hardware through floating-point
  differences (`get_device` enables TF32 matmuls on CUDA, one such source): expect the shape
  and the ranking to reproduce, not every digit.
- *n* is small (160 in `FULL`); adjacent rungs within a couple of points are not resolvable.

What the page may and may not claim from this run — and the full limitations list — is
[`docs/page_5_refactor.md`](../docs/page_5_refactor.md) §6, which the translation step checks
itself against ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).